In [4]:
# ============================================================
# H2: LABELING — UPDATED (4 Rules, Aggressive Philosophy)
#
# Rule 1: Seller Buyback < 30 days
#         Liu et al. (2023) arxiv:2305.01543
#
# Rule 2: Identity Trade (Self-Trade)
#         La Morgia et al. (2023), Oh (2024)
#
# Rule 3: Multi-hop Cycle < 7 days
#         Oh (2024), Von Wachter et al. (2022)
#
# Rule 4: High Transaction Count per Wallet Pair
#         Niu et al. (ACM WWW 2024)
#
# Philosophy: Aggressive — prefer FP over FN
# ============================================================

import sqlite3
import pandas as pd
import numpy as np

# Load raw data (before feature engineering)
# Kita perlu raw data karena labeling harus dari awal
conn = sqlite3.connect('../../dataset/nfts.sqlite/nfts.sqlite')

print("Loading raw transfers (April - September 2021)...")

APRIL_1 = 1617235200   # 2021-04-01
OCT_1   = 1633046400   # 2021-10-01

transfers_raw = pd.read_sql_query(f"""
    SELECT
        transaction_hash,
        timestamp,
        nft_address,
        token_id,
        from_address,
        to_address,
        transaction_value
    FROM transfers
    WHERE timestamp >= {APRIL_1}
      AND timestamp <  {OCT_1}
""", conn)

print(f"Loaded: {len(transfers_raw):,} rows")

# Fix types
transfers_raw['timestamp_dt'] = pd.to_datetime(
    transfers_raw['timestamp'], unit='s'
)
transfers_raw['transaction_value'] = pd.to_numeric(
    transfers_raw['transaction_value'], errors='coerce'
).fillna(0.0)

# Filter sales only (value > 0)
sales = transfers_raw[
    transfers_raw['transaction_value'] > 0
].copy()
print(f"Sales only (value > 0): {len(sales):,}")

# Remove burn addresses
BURN = {
    '0x0000000000000000000000000000000000000000',
    '0x000000000000000000000000000000000000dead'
}
burn_lower = {a.lower() for a in BURN}

sales_clean = sales[
    ~sales['from_address'].str.lower().isin(burn_lower) &
    ~sales['to_address'].str.lower().isin(burn_lower)
].copy().reset_index(drop=True)

print(f"After burn filter: {len(sales_clean):,}")

# Sort by timestamp — critical for temporal rules
sales_clean = sales_clean.sort_values(
    'timestamp'
).reset_index(drop=True)

# Initialize
sales_clean['is_wash_trading'] = 0
wash_hashes = set()

# Constants
MAX_SECS_BUYBACK  = 30 * 24 * 3600  # Rule 1: 30 days
MAX_SECS_CYCLE    = 7  * 24 * 3600  # Rule 3: 7 days
MAX_HOPS          = 10              # Rule 3: max hops
PAIR_TX_THRESHOLD = 3               # Rule 4: min tx count

grouped      = sales_clean.groupby(['nft_address', 'token_id'])
total_groups = len(grouped)

print(f"\nLabeling rules:")
print(f"  Rule 1 (Liu 2023)      : Seller buyback < 30 days")
print(f"  Rule 2 (La Morgia 2023): Identity trade (self-trade)")
print(f"  Rule 3 (Oh 2024)       : Multi-hop cycle < 7 days")
print(f"  Rule 4 (Niu 2024)      : High tx count per wallet pair >= {PAIR_TX_THRESHOLD}")
print(f"\nUnique NFT tokens: {total_groups:,}")
print(f"Processing Rules 1, 2, 3...")

r1_count = 0
r2_count = 0
r3_count = 0
processed = 0

for (nft_addr, token_id), group in grouped:

    if len(group) < 1:
        processed += 1
        continue

    group = group.sort_values(
        'timestamp'
    ).reset_index(drop=True)
    n = len(group)

    for i in range(n):
        from_i = group.loc[i, 'from_address'].lower()
        to_i   = group.loc[i, 'to_address'].lower()
        hash_i = group.loc[i, 'transaction_hash']
        ts_i   = group.loc[i, 'timestamp']

        # ------------------------------------------------
        # RULE 2: Identity Trade
        # La Morgia et al. (2023), Oh (2024)
        # Seller address == Buyer address
        # ------------------------------------------------
        if from_i == to_i:
            wash_hashes.add(hash_i)
            r2_count += 1
            continue  # no need to check further for this tx

        # Chain tracking for Rule 3
        chain_hashes = [hash_i]
        chain_end    = to_i

        for j in range(i + 1, n):
            ts_j      = group.loc[j, 'timestamp']
            diff_secs = ts_j - ts_i

            # ----------------------------------------
            # RULE 1: Seller Buyback < 30 days
            # Liu et al. (2023)
            # ----------------------------------------
            if diff_secs > MAX_SECS_BUYBACK:
                break

            from_j = group.loc[j, 'from_address'].lower()
            to_j   = group.loc[j, 'to_address'].lower()
            hash_j = group.loc[j, 'transaction_hash']

            # A sells → ... → A buys back
            if from_i == to_j:
                wash_hashes.add(hash_i)
                wash_hashes.add(hash_j)
                r1_count += 1

            # ----------------------------------------
            # RULE 3: Multi-hop Cycle < 7 days
            # Oh (2024), Von Wachter et al. (2022)
            # A→B→C→...→A within 7 days
            # ----------------------------------------
            if diff_secs <= MAX_SECS_CYCLE:
                if (from_j == chain_end and
                        len(chain_hashes) < MAX_HOPS):
                    chain_hashes.append(hash_j)
                    chain_end = to_j

                    if to_j == from_i:
                        for h in chain_hashes:
                            wash_hashes.add(h)
                        r3_count += 1
                        # Reset chain
                        chain_hashes = [hash_i]
                        chain_end    = to_i

    processed += 1
    if processed % 100000 == 0:
        print(f"  Progress: {processed:,} / "
              f"{total_groups:,} "
              f"({processed/total_groups:.1%})")

print(f"\nRule 1 (seller buyback)  : {r1_count:,} pairs")
print(f"Rule 2 (identity trade)  : {r2_count:,} transactions")
print(f"Rule 3 (multi-hop cycle) : {r3_count:,} cycles")
print(f"Total hashes so far      : {len(wash_hashes):,}")

# ============================================================
# RULE 4: High Transaction Count per Wallet Pair
# Niu et al. (ACM WWW 2024)
# Wallet pair (A, B) that traded the same NFT
# more than threshold times
# ============================================================
print(f"\nProcessing Rule 4 (High tx count per wallet pair)...")

r4_count     = 0
before_rule4 = len(wash_hashes)

# Count transactions per (from, to, nft_address, token_id)
pair_counts = sales_clean.groupby([
    'from_address', 'to_address',
    'nft_address', 'token_id'
]).size().reset_index(name='tx_count')

# Flag pairs exceeding threshold
suspicious_pairs = pair_counts[
    pair_counts['tx_count'] >= PAIR_TX_THRESHOLD
]

print(f"  Suspicious wallet pairs: {len(suspicious_pairs):,}")

for _, row in suspicious_pairs.iterrows():
    mask = (
        (sales_clean['from_address'] == row['from_address']) &
        (sales_clean['to_address']   == row['to_address']) &
        (sales_clean['nft_address']  == row['nft_address']) &
        (sales_clean['token_id']     == row['token_id'])
    )
    new_hashes = set(
        sales_clean[mask]['transaction_hash'].tolist()
    ) - wash_hashes
    wash_hashes.update(new_hashes)
    r4_count += len(new_hashes)

print(f"  New hashes from Rule 4 : {r4_count:,}")

# ============================================================
# APPLY LABELS
# ============================================================
sales_clean.loc[
    sales_clean['transaction_hash'].isin(wash_hashes),
    'is_wash_trading'
] = 1

# ============================================================
# RESULTS
# ============================================================
total     = len(sales_clean)
wt_count  = int(sales_clean['is_wash_trading'].sum())
norm_count = total - wt_count
ratio     = wt_count / norm_count

print(f"\n{'='*55}")
print(f"LABELING RESULTS (4 Rules)")
print(f"{'='*55}")
print(f"Total sales      : {total:,}")
print(f"Wash trading (1) : {wt_count:,} ({wt_count/total:.3%})")
print(f"Normal (0)       : {norm_count:,}")
print(f"Ratio            : 1 : {norm_count//wt_count}")
print(f"\nBreakdown:")
print(f"  Rule 1 (seller buyback)  : {r1_count:,} pairs")
print(f"  Rule 2 (identity trade)  : {r2_count:,} tx")
print(f"  Rule 3 (multi-hop cycle) : {r3_count:,} cycles")
print(f"  Rule 4 (high pair tx)    : {r4_count:,} new tx")

# Sanity check
wt_df      = sales_clean[sales_clean['is_wash_trading'] == 1]
burn_in_wt = wt_df[
    wt_df['to_address'].str.lower().isin(burn_lower)
]
print(f"\nSanity check:")
print(f"  Burn addresses in labels: {len(burn_in_wt)}")
print(f"  (Expected: 0)")

# Check if ratio is within 1:50
if norm_count // wt_count <= 50:
    print(f"\n✅ Ratio {norm_count//wt_count}:1 — within 1:50 limit!")
else:
    print(f"\n⚠️ Ratio {norm_count//wt_count}:1 — exceeds 1:50!")
    print(f"   Need to undersample normal class")
    print(f"   Target normal: {wt_count * 50:,}")
    print(f"   Need to remove: {norm_count - wt_count*50:,}")

Loading raw transfers (April - September 2021)...
Loaded: 4,514,729 rows
Sales only (value > 0): 2,748,291
After burn filter: 2,713,386

Labeling rules:
  Rule 1 (Liu 2023)      : Seller buyback < 30 days
  Rule 2 (La Morgia 2023): Identity trade (self-trade)
  Rule 3 (Oh 2024)       : Multi-hop cycle < 7 days
  Rule 4 (Niu 2024)      : High tx count per wallet pair >= 3

Unique NFT tokens: 1,979,238
Processing Rules 1, 2, 3...
  Progress: 100,000 / 1,979,238 (5.1%)
  Progress: 200,000 / 1,979,238 (10.1%)
  Progress: 300,000 / 1,979,238 (15.2%)
  Progress: 400,000 / 1,979,238 (20.2%)
  Progress: 500,000 / 1,979,238 (25.3%)
  Progress: 600,000 / 1,979,238 (30.3%)
  Progress: 700,000 / 1,979,238 (35.4%)
  Progress: 800,000 / 1,979,238 (40.4%)
  Progress: 900,000 / 1,979,238 (45.5%)
  Progress: 1,000,000 / 1,979,238 (50.5%)
  Progress: 1,100,000 / 1,979,238 (55.6%)
  Progress: 1,200,000 / 1,979,238 (60.6%)
  Progress: 1,300,000 / 1,979,238 (65.7%)
  Progress: 1,400,000 / 1,979,238 (70.7%)

In [5]:
sales_clean.to_csv(
    "labeled_transactions_april_sept.csv",
    index=False
)

In [6]:
# ============================================================
# FULL TGN FEASIBILITY AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("="*70)
print("DATASET SUMMARY")
print("="*70)

total_tx = len(sales_clean)

fraud_df = sales_clean[
    sales_clean['is_wash_trading'] == 1
]

fraud_tx = len(fraud_df)

fraud_wallets = set(
    fraud_df['from_address']
).union(
    set(fraud_df['to_address'])
)

print(f"Total Transactions : {total_tx:,}")
print(f"Fraud Transactions : {fraud_tx:,}")
print(f"Fraud Ratio        : {fraud_tx/total_tx:.4%}")
print(f"Fraud Wallets      : {len(fraud_wallets):,}")


# ============================================================
# AUDIT 1
# WALLET ACTIVITY
# ============================================================

print("\n")
print("="*70)
print("AUDIT 1 — WALLET ACTIVITY")
print("="*70)

sender_counts = sales_clean.groupby(
    'from_address'
).size()

receiver_counts = sales_clean.groupby(
    'to_address'
).size()

wallet_activity = sender_counts.add(
    receiver_counts,
    fill_value=0
).astype(int)

print(wallet_activity.describe())

for p in [50,75,90,95,99]:
    print(
        f"P{p}: "
        f"{wallet_activity.quantile(p/100):.0f}"
    )


# ============================================================
# AUDIT 2
# FRAUD WALLET HISTORY
# ============================================================

print("\n")
print("="*70)
print("AUDIT 2 — FRAUD WALLET HISTORY")
print("="*70)

fraud_wallet_activity = wallet_activity[
    wallet_activity.index.isin(fraud_wallets)
]

print(
    fraud_wallet_activity.describe()
)

for p in [50,75,90,95,99]:
    print(
        f"P{p}: "
        f"{fraud_wallet_activity.quantile(p/100):.0f}"
    )


# ============================================================
# AUDIT 3
# NFT CHAIN LENGTH
# ============================================================

print("\n")
print("="*70)
print("AUDIT 3 — NFT CHAIN LENGTH")
print("="*70)

nft_chain_lengths = sales_clean.groupby(
    ['nft_address', 'token_id']
).size()

print(
    nft_chain_lengths.describe()
)

for p in [50,75,90,95,99]:
    print(
        f"P{p}: "
        f"{nft_chain_lengths.quantile(p/100):.0f}"
    )


# ============================================================
# AUDIT 4
# FRAUD INTERACTIONS PER WALLET
# ============================================================

print("\n")
print("="*70)
print("AUDIT 4 — FRAUD INTERACTIONS PER WALLET")
print("="*70)

fraud_tx_per_wallet = pd.concat([
    fraud_df['from_address'],
    fraud_df['to_address']
]).value_counts()

print(
    fraud_tx_per_wallet.describe()
)

for p in [50,75,90,95,99]:
    print(
        f"P{p}: "
        f"{fraud_tx_per_wallet.quantile(p/100):.0f}"
    )


# ============================================================
# AUDIT 5
# FRAUD WALLET LIFESPAN
# ============================================================

print("\n")
print("="*70)
print("AUDIT 5 — FRAUD WALLET LIFESPAN")
print("="*70)

wallet_events = pd.concat([
    sales_clean[['from_address','timestamp']]
        .rename(columns={'from_address':'wallet'}),
    sales_clean[['to_address','timestamp']]
        .rename(columns={'to_address':'wallet'})
])

wallet_events = wallet_events[
    wallet_events['wallet'].isin(fraud_wallets)
]

wallet_span = wallet_events.groupby(
    'wallet'
)['timestamp'].agg(['min','max'])

wallet_span['lifespan_days'] = (
    wallet_span['max'] -
    wallet_span['min']
) / 86400

print(
    wallet_span['lifespan_days'].describe()
)

for p in [50,75,90,95,99]:
    print(
        f"P{p}: "
        f"{wallet_span['lifespan_days'].quantile(p/100):.1f} days"
    )


# ============================================================
# TGN READINESS SCORE (heuristic)
# ============================================================

print("\n")
print("="*70)
print("TGN READINESS CHECK")
print("="*70)

fraud_history_median = fraud_wallet_activity.quantile(0.5)
lifespan_median = wallet_span['lifespan_days'].quantile(0.5)

print(f"Median Fraud Wallet Activity : {fraud_history_median:.0f}")
print(f"Median Fraud Wallet Lifespan : {lifespan_median:.1f} days")

if fraud_history_median >= 20 and lifespan_median >= 30:
    print("\n✅ STRONG TEMPORAL SIGNAL")
    print("TGN appears reasonably justified.")
elif fraud_history_median >= 10 and lifespan_median >= 14:
    print("\n⚠️ MODERATE TEMPORAL SIGNAL")
    print("TGN may help, but benefit is uncertain.")
else:
    print("\n❌ WEAK TEMPORAL SIGNAL")
    print("GraphEdgeClassifier may be sufficient.")

print("\nAudit complete.")

DATASET SUMMARY
Total Transactions : 2,713,386
Fraud Transactions : 18,953
Fraud Ratio        : 0.6985%
Fraud Wallets      : 3,543


AUDIT 1 — WALLET ACTIVITY
count    333077.000000
mean         16.292845
std         253.300586
min           1.000000
25%           1.000000
50%           2.000000
75%           9.000000
max      139242.000000
dtype: float64
P50: 2
P75: 9
P90: 30
P95: 62
P99: 229


AUDIT 2 — FRAUD WALLET HISTORY
count     3543.000000
mean       153.226926
std        490.516342
min          1.000000
25%          6.000000
50%         34.000000
75%        135.000000
max      18742.000000
dtype: float64
P50: 34
P75: 135
P90: 382
P95: 681
P99: 1599


AUDIT 3 — NFT CHAIN LENGTH
count    1.979238e+06
mean     1.370925e+00
std      7.473031e-01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      5.300000e+01
dtype: float64
P50: 1
P75: 2
P90: 2
P95: 3
P99: 4


AUDIT 4 — FRAUD INTERACTIONS PER WALLET
count    3543.000000
mean       10.69

In [7]:
# ============================================================
# BUILD TEMPORAL WALLET-WALLET GRAPH
# For GraphEdgeClassifier / TGN
# ============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

print("="*70)
print("BUILDING TEMPORAL GRAPH")
print("="*70)

# ------------------------------------------------------------
# 1. Sort by time
# ------------------------------------------------------------
sales_clean = sales_clean.sort_values(
    "timestamp"
).reset_index(drop=True)

# ------------------------------------------------------------
# 2. Encode wallet addresses
# ------------------------------------------------------------
wallet_encoder = LabelEncoder()

all_wallets = pd.concat([
    sales_clean["from_address"],
    sales_clean["to_address"]
]).unique()

wallet_encoder.fit(all_wallets)

sales_clean["src"] = wallet_encoder.transform(
    sales_clean["from_address"]
)

sales_clean["dst"] = wallet_encoder.transform(
    sales_clean["to_address"]
)

# ------------------------------------------------------------
# 3. Select edge features
# ------------------------------------------------------------
candidate_features = [
    "transaction_value",
    "holding_time_hours",
    "pair_frequency",
    "time_since_last_hours",
    "symmetry_ratio_from",
    "transfers_out_from",
    "transfers_in_from",
    "transfers_out_to",
    "transfers_in_to",
    "num_transitions",
    "src_tx_count_past",
    "dst_tx_count_past",
    "src_to_dst_count_past",
    "dst_to_src_count_past",
    "time_since_src_last",
    "time_since_dst_last",
    "log_time_since_src_last",
    "log_time_since_dst_last",
    "src_value_sum_past",
    "dst_value_sum_past",
    "mint_age_seconds",
    "log_transaction_value"
]

edge_features = [
    col for col in candidate_features
    if col in sales_clean.columns
]

print(f"Edge features found: {len(edge_features)}")

# ------------------------------------------------------------
# 4. Missing values
# ------------------------------------------------------------
sales_clean[edge_features] = (
    sales_clean[edge_features]
    .fillna(0)
)

# ------------------------------------------------------------
# 5. Edge feature matrix
# ------------------------------------------------------------
edge_feat = (
    sales_clean[edge_features]
    .astype(np.float32)
    .values
)

# ------------------------------------------------------------
# 6. Labels
# ------------------------------------------------------------
labels = (
    sales_clean["is_wash_trading"]
    .astype(np.int64)
    .values
)

# ------------------------------------------------------------
# 7. Temporal split
# ------------------------------------------------------------
n = len(sales_clean)

train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_df = sales_clean.iloc[:train_end]
val_df   = sales_clean.iloc[train_end:val_end]
test_df  = sales_clean.iloc[val_end:]

# ------------------------------------------------------------
# 8. Statistics
# ------------------------------------------------------------
fraud_wallets = set(
    sales_clean.loc[
        sales_clean["is_wash_trading"] == 1,
        "src"
    ]
).union(
    set(
        sales_clean.loc[
            sales_clean["is_wash_trading"] == 1,
            "dst"
        ]
    )
)

print("\n" + "="*70)
print("GRAPH SUMMARY")
print("="*70)

print(f"Nodes              : {sales_clean['src'].nunique():,}")
print(f"Edges              : {len(sales_clean):,}")
print(f"Fraud edges        : {labels.sum():,}")
print(f"Fraud ratio        : {labels.mean():.4%}")
print(f"Fraud wallets      : {len(fraud_wallets):,}")

print("\nTemporal Split")
print(f"Train              : {len(train_df):,}")
print(f"Validation         : {len(val_df):,}")
print(f"Test               : {len(test_df):,}")

print("\nFraud per split")
print(
    f"Train fraud        : "
    f"{train_df['is_wash_trading'].sum():,}"
)

print(
    f"Validation fraud   : "
    f"{val_df['is_wash_trading'].sum():,}"
)

print(
    f"Test fraud         : "
    f"{test_df['is_wash_trading'].sum():,}"
)

print("\nEdge feature shape")
print(edge_feat.shape)

print("\nDone.")

BUILDING TEMPORAL GRAPH
Edge features found: 1

GRAPH SUMMARY
Nodes              : 134,960
Edges              : 2,713,386
Fraud edges        : 18,953
Fraud ratio        : 0.6985%
Fraud wallets      : 3,543

Temporal Split
Train              : 1,899,370
Validation         : 407,008
Test               : 407,008

Fraud per split
Train fraud        : 15,818
Validation fraud   : 1,684
Test fraud         : 1,451

Edge feature shape
(2713386, 1)

Done.


In [6]:
print(sorted(sales_clean.columns))

['dst', 'from_address', 'is_wash_trading', 'nft_address', 'src', 'timestamp', 'timestamp_dt', 'to_address', 'token_id', 'transaction_hash', 'transaction_value']
